# Home Energy Planner — Execution Notebook (Google Colab)

This notebook runs the **Home Energy Planner** project end to end and keeps the outputs of that run:
a neural network forecasts household electricity load, a **CSP** (Backtracking, Forward Checking, AC-3) schedules flexible appliances, and **Uniform Cost Search / A\*** plans a home battery.

> **Real vs simulated.** Electricity consumption comes from the real UCI *Individual Household Electric Power Consumption* dataset. Prices, appliance profiles, battery specification and power limits are **simulated parameters**, so every cost shown is a **simulated** result, not a measured saving.

## Contents
1. Environment setup
2. Data preparation and data dictionary
3. Neural network training (vs baselines)
4. Experiments (3 experiments, 5 seeds each)
5. Integrated demo
6. Tests
7. Report generation
8. Documentation files
9. Project output overview
10. Version control (git) check
11. Summary of key results
12. Appendix: resuming a session

**How to run:** execute the cells from top to bottom. The first cell asks you to upload `home_energy_planner_project.zip`.

## 1. Environment Setup

### 1.1 Mount Google Drive and extract the project
Mounts Drive, asks for the project ZIP, and extracts it to `MyDrive/Home_Energy_Planner` so files survive between sessions.

In [ ]:

from google.colab import drive, files
import os
import zipfile

# Connect Google Drive
drive.mount('/content/drive')

# Upload the project ZIP
uploaded = files.upload()

# Extract the project into Google Drive
zip_name = next(iter(uploaded))
project_dir = "/content/drive/MyDrive/Home_Energy_Planner"

os.makedirs(project_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(project_dir)

print("Project extracted successfully!")
print("Project files:")

for item in os.listdir(project_dir):
    print("-", item)


Mounted at /content/drive


Saving home_energy_planner_project.zip to home_energy_planner_project.zip
Project extracted successfully!
Project files:
- train_nn.py
- CONTRIBUTIONS.md
- requirements.txt
- planner
- pytest.ini
- README.md
- make_report.py
- run_experiments.py
- export_git_log.sh
- .gitignore
- run_demo.py
- tests
- home_energy_planner.ipynb
- make_data.py
- docs


### 1.2 Check the working directory
Switches to the project folder and confirms that the key files (`requirements.txt`, `make_data.py`, `planner/`) exist. All following cells use paths relative to this folder.

In [ ]:

import sys
import os

project_dir = "/content/drive/MyDrive/Home_Energy_Planner"

print("Python version:", sys.version)
print("Current directory:", os.getcwd())
print("Project directory exists:", os.path.exists(project_dir))

os.chdir(project_dir)

print("\nWorking directory:", os.getcwd())
print("\nRequirements file exists:", os.path.exists("requirements.txt"))
print("Data script exists:", os.path.exists("make_data.py"))
print("Planner folder exists:", os.path.isdir("planner"))


Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Current directory: /content
Project directory exists: True

Working directory: /content/drive/MyDrive/Home_Energy_Planner

Requirements file exists: True
Data script exists: True
Planner folder exists: True


### 1.3 Dependencies
Shows the pinned `requirements.txt` and the Python version mentioned in the README, then checks which packages are installed in this Colab runtime. Pinned versions are the ones the project was tested with; Colab may have slightly different versions, which is fine as long as the tests pass.

In [ ]:
# Display project dependencies
with open("requirements.txt", "r") as f:
    print(f.read())

# Display project Python version requirements, if documented
with open("README.md", "r") as f:
    readme = f.read()

for line in readme.splitlines():
    if "python" in line.lower():
        print(line)

# Pinned versions = the versions this package was tested with (Python 3.12).
# Other recent versions will normally work; if something differs, install exactly these.
numpy==2.5.4
pandas==3.0.6
matplotlib==3.11.2
scikit-learn==1.9.1
torch==2.15.0
pytest==9.2.0
python-docx==1.2.0

## Environment setup (tested with Python 3.12)
python -m venv .venv
python make_data.py                  # 1. download + preprocess -> data/processed/ (daily_matrix.csv, summary.json, data_dictionary.csv, split_info.csv)
python train_nn.py                   # 2. train NN, test results vs baselines -> results/nn_vs_baselines.csv, models/forecaster.pt
python run_experiments.py            # 3. experiments 1-3 -> results/*.csv and *.png   (a few minutes)
python run_demo.py                   # 4. integrated demo (plots saved to outputs/; add --no-show on a server)
python make_report.py --authors "Name1, Name2, Name3"   # 6. report from the real results -> report/*.docx


In [ ]:
import importlib
import sys

print("Python:", sys.version)

packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "sklearn",
    "torch",
    "pytest",
    "docx"
]

for package in packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "Version not available")
        print(f"{package}: INSTALLED ({version})")
    except ImportError:
        print(f"{package}: NOT INSTALLED")

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
numpy: INSTALLED (2.4.6)
pandas: INSTALLED (2.2.3)
matplotlib: INSTALLED (3.10.0)
sklearn: INSTALLED (1.6.1)
torch: INSTALLED (2.14.1+cpu)
pytest: INSTALLED (8.4.2)
docx: NOT INSTALLED


### 1.4 Install the missing package
`python-docx` is the only package not preinstalled in Colab; it is needed by `make_report.py`.

In [ ]:
%pip install python-docx==1.2.0

In [ ]:
import docx
print("python-docx version:", docx.__version__)

python-docx version: 1.2.0


### 1.5 Check that the project modules import
Imports every module of the `planner` package (config, data, nn, csp, search, integration, evaluation, plotting) to catch path or dependency problems before running anything long.

In [ ]:
import planner.config
import planner.data
import planner.nn
import planner.csp
import planner.search
import planner.integration
import planner.evaluation
import planner.plotting

print("All planner modules imported successfully!")

All planner modules imported successfully!


## 2. Data Preparation

`make_data.py` downloads the UCI dataset (if needed), parses the timestamps, averages the minute data to **hourly kW**, forward-fills gaps of at most 3 hours (causal, no future information), drops days that still have gaps, and splits the days **chronologically** (final 365 days = test, the 90 days before = validation, the rest = training; no shuffling). The processed data and a summary are written to `data/processed/`.

Read the output as: raw minute rows and share missing → hours missing / filled → valid vs invalid days → number of samples in each split.

In [ ]:
!python make_data.py

[data] processed data written to data/processed/
Summary:
  minute_rows: 2075259
  missing_minute_share: 0.012518437457686004
  hours_total: 34608
  hours_missing_before_fill: 451
  hours_filled: 9
  hours_still_missing: 552
  days_total: 1442
  days_valid: 1419
  days_invalid: 23
  first_timestamp: 2006-12-16 17:24:00
  last_timestamp: 2010-11-26 21:02:00
  seed: 42
Samples: train=962 (2006-12-24..2009-08-28), val=90 (2009-08-29..2009-11-26), test=331 (2009-11-27..2010-11-26)


### Data dictionary
The data dictionary lists every column or item, its meaning, unit, whether it is **real or generated**, and **how generated parts were made** (`data/processed/data_dictionary.csv`). First the file is located, then displayed.

In [ ]:
%cd /content/drive/MyDrive/Home_Energy_Planner

import os

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    for file in files:
        if any(word in file.lower() for word in ["dictionary", "data", "schema"]):
            print(os.path.join(root, file))

/content
./make_data.py
./planner/data.py
./tests/test_data.py
./data/processed/data_dictionary.csv


In [ ]:
import pandas as pd

path = "data/processed/data_dictionary.csv"
df = pd.read_csv(path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df)

Shape: (19, 5)
Columns: ['column / item', 'meaning', 'unit', 'real or generated', 'how generated / derived']


,column / item,meaning,unit,real or generated,how generated / derived
0,Date,Calendar date dd/mm/YYYY of the measurement,date,REAL (UCI raw),measured by the household logger
1,Time,Time hh:mm:ss of the measurement,time,REAL (UCI raw),measured
2,Global_active_power,Household minute-averaged active power (source...,kW,"REAL (UCI raw), used",measured; '?' = missing
3,Global_reactive_power,Household minute-averaged reactive power,kW,"REAL (UCI raw), not used",measured
4,Voltage,Minute-averaged voltage,V,"REAL (UCI raw), not used",measured
5,Global_intensity,Minute-averaged current intensity,A,"REAL (UCI raw), not used",measured
6,Sub_metering_1/2/3,Energy sub-meters (kitchen / laundry / water h...,Wh,"REAL (UCI raw), not used",measured
7,load_kw,Hourly mean of Global_active_power over valid ...,kW,DERIVED from real data,resample('h').mean(); NaN if < 30 valid minute...
8,valid_minutes,Number of valid minutes inside the hour (hour ...,count,DERIVED from real data,resample('h').count()
9,prev_day_h00..h23,Hourly load of target day - 1,kW,DERIVED from real data,daily matrix row D-1


## 3. Neural Network Training

`train_nn.py` trains the forecaster (MLP 53 → 128 → 64 → 24 with ReLU and dropout, MSE loss, AdamW, early stopping on validation loss) and evaluates it on the test set against two baselines: **yesterday (same hour)** and **same hour last week**. All three predict exactly the same test days.

The table reports MAE, RMSE and MAPE (MAPE counts only actual loads of at least 0.2 kW). Lower is better.

In [ ]:
!python train_nn.py

[data] loaded cached daily matrix data/processed/daily_matrix.csv (1442 days)
epochs run: 55, best epoch: 35
                       MAE (kW)  RMSE (kW)  MAPE (%)
Neural Network           0.4507     0.6216   58.6737
Yesterday (same hour)    0.5721     0.8412   69.6320
Last week (same hour)    0.5865     0.8473   74.2822


### Check the saved artifacts
Confirms that the trained model (`models/forecaster.pt`) and the comparison table (`results/nn_vs_baselines.csv`) were written.

In [ ]:
import os

files_to_check = [
    "models/forecaster.pt",
    "results/nn_vs_baselines.csv"
]

for path in files_to_check:
    print(path, "->", "EXISTS" if os.path.exists(path) else "MISSING")

models/forecaster.pt -> EXISTS
results/nn_vs_baselines.csv -> EXISTS


## 4. Experiments

`run_experiments.py` runs three experiments, each with **5 seeds (42–46)**, and reports **mean ± standard deviation**:

| # | Experiment | What is compared |
|---|---|---|
| 1 | **UCS vs A\*** | cost, nodes expanded/generated, time, agreement with the dynamic-programming optimum, heuristic admissibility |
| 2 | **BT vs BT+FC vs BT+FC+AC3** | assignments, rejected values, backtracks, domain reductions, time, for 3 / 5 / 8 appliances (normal and tight household limit) |
| 3 | **NN vs baselines inside the planner** | forecast error, simulated cost per day, household-limit and grid-limit violations; an *Oracle* row (real load as forecast) is a reference only |

Tables are printed below and saved with plots in `results/`.

In [ ]:
!python run_experiments.py

[data] loaded cached daily matrix data/processed/daily_matrix.csv (1442 days)

=== Experiment 1: UCS vs A* (30 scenarios per run, 5 runs) ===
          cost EUR (simulated) nodes expanded nodes generated          time ms
algorithm                                                                     
A*              4.694 +- 0.404  203.3 +- 2.53    1112 +- 23.1  4.445 +- 0.0565
UCS             4.694 +- 0.404  220.2 +- 3.03    1216 +- 26.5   4.431 +- 0.112
cost equals DP optimum in all runs: True | heuristic admissible on every state: True

=== Experiment 2: BT vs BT+FC vs BT+FC+AC3 (3, 5, 8 appliances) ===
                                                      time ms   assignments      rejected    backtracks domain reductions
scenario              appliances algorithm                                                                               
normal limit (5.0 kW) 3          BT          0.1058 +- 0.0282        3 +- 0  1.2 +- 0.447        0 +- 0            0 +- 0
                      

### Files produced by the experiments
Lists the contents of `results/` (CSV tables and PNG plots) and `models/`.

In [ ]:
import os

print("Results folder:")
for item in sorted(os.listdir("results")):
    print("-", item)

print("\nModels folder:")
for item in sorted(os.listdir("models")):
    print("-", item)

Results folder:
- exp1_runs.csv
- exp1_summary.csv
- exp1_ucs_vs_astar.png
- exp2_csp_comparison.png
- exp2_runs.csv
- exp2_summary.csv
- exp3_nn_vs_baselines.png
- exp3_runs.csv
- exp3_summary.csv
- nn_forecast_vs_actual.png
- nn_training_loss.png
- nn_vs_baselines.csv

Models folder:
- forecaster.pt


## 5. Integrated Demo

`run_demo.py` shows how the three components cooperate on a real test-period day (the network never trained on it). The numbered sections of the output are:

1. **Integrated run** — input history → NN forecast → CSP schedule → battery search → simulated cost and constraint checks
2. **CSP pruning** — AC-3 domain reduction, then BT vs BT+FC vs BT+FC+AC3 on a hard 8-appliance instance
3. **Search trace** — UCS vs A\* expansions on the same battery problem
4. **Feedback loop** — a synthetic stress scenario where the battery plan fails, a clashing appliance pair is forbidden and the CSP re-runs
5. **No-solution message** — clear explanation and relaxation offer
6. **NN ON vs OFF** — same 30 test days; OFF means a *yesterday persistence* forecast

Plots are saved to `outputs/` (`--no-show` keeps Colab from blocking). All costs are **simulated**.

In [ ]:
!python run_demo.py --no-show


0) Data and model
[data] loaded cached daily matrix data/processed/daily_matrix.csv (1442 days)
[model] loaded models/forecaster.pt
Demo day (real test-period day, never used for training): 2010-02-17

1) INTEGRATED RUN: input -> forecast -> CSP -> battery search -> plan
=== Home Energy Planner - 2010-02-17 (all costs SIMULATED) ===
1) INPUT history (real): day-1 mean 1.25 kW, day-7 mean 1.35 kW
2) NN forecast: mean 1.25 kW, peak 2.25 kW at 20:00
3) CSP (BT+FC+AC3): assignments=5, backtracks=0, domain reductions=12
      appliance start   end  power kW window
washing_machine 07:00 09:00       2.0  07-22
          dryer 21:00 23:00       2.5  08-23
     dishwasher 12:00 14:00       1.8  12-24
     ev_charger 00:00 04:00       3.3  00-24
  electric_oven 11:00 12:00       2.2  11-21
4) Battery search (A*): expanded=213, generated=1140, time=2.6 ms
5) Feedback iterations used: 1 (no battery failure occurred)
 hour    action  kWh  battery after  grid kWh  price  cost EUR
    0      idle  0

## 6. Tests

The pytest suite covers preprocessing (including a no-future-leakage check), forecast shape and validity, search transitions and a hand-solved known-answer example, CSP constraints, Forward Checking and AC-3 pruning, battery capacity boundaries, invalid inputs, and no-solution messages. Tests are tagged **normal**, **edge** and **impossible**.

First make sure the project folder is on the Python path, then run all tests.

In [ ]:
%cd /content/drive/MyDrive/Home_Energy_Planner

import os
import sys

print("Current directory:", os.getcwd())
print("Planner folder exists:", os.path.isdir("planner"))

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import planner
print("Planner imported successfully!")

/content
Current directory: /content/drive/MyDrive/Home_Energy_Planner
Planner folder exists: True
Planner imported successfully!


In [ ]:
!python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/Home_Energy_Planner
configfile: pytest.ini
testpaths: tests
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 29 items                                                             

tests/test_csp.py::test_csp_known_unique_solution PASSED                 [  3%]
tests/test_csp.py::test_all_csp_variants_return_valid_schedules PASSED   [  6%]
tests/test_csp.py::test_forward_checking_and_ac3_prune_domains PASSED    [ 10%]
tests/test_csp.py::test_shared_circuit_appliances_do_not_overlap PASSED  [ 13%]
tests/test_csp.py::test_edge_no_appliances_gives_empty_schedule PASSED   [ 17%]
tests/test_csp.py::test_edge_invalid_inputs_are_rejected PASSED          [ 20%]
tests/test_csp.py::test_impossible_dryer_cannot_follow_washer PASSED     [ 24%]
te

### Test inventory
Lists the collected tests per file (without running them).

In [ ]:
%cd /content/drive/MyDrive/Home_Energy_Planner

!python -m pytest --collect-only -q

/content
tests/test_csp.py::test_csp_known_unique_solution
tests/test_csp.py::test_all_csp_variants_return_valid_schedules
tests/test_csp.py::test_forward_checking_and_ac3_prune_domains
tests/test_csp.py::test_shared_circuit_appliances_do_not_overlap
tests/test_csp.py::test_edge_no_appliances_gives_empty_schedule
tests/test_csp.py::test_edge_invalid_inputs_are_rejected
tests/test_csp.py::test_impossible_dryer_cannot_follow_washer
tests/test_csp.py::test_impossible_household_limit_below_forecast_everywhere
tests/test_data.py::test_hourly_aggregation_and_gap_filling
tests/test_data.py::test_only_short_gaps_are_filled_and_bad_days_dropped
tests/test_data.py::test_chronological_split_is_ordered_and_disjoint
tests/test_data.py::test_features_use_no_future_information
tests/test_data.py::test_real_data_split_is_chronological
tests/test_forecast.py::test_forecast_shape_and_validity
tests/test_forecast.py::test_metrics_match_sklearn
tests/test_forecast.py::test_edge_mape_ignores_near_zero_actu

## 7. Report Generation

`make_report.py` builds the project report (`.docx`) from the **real result files** in `results/`, so every number in the report comes from the experiments above. First the available options, then the build.

In [ ]:
!python make_report.py --help

usage: make_report.py [-h] [--results RESULTS] [--outputs OUTPUTS]
                      [--authors AUTHORS] [--out OUT]

options:
  -h, --help         show this help message and exit
  --results RESULTS
  --outputs OUTPUTS
  --authors AUTHORS
  --out OUT


In [ ]:
!python make_report.py

saved report/Home_Energy_Planner_Report.docx


## 8. Documentation Files

### 8.1 README
Setup instructions, exact run commands, project structure, and the real-vs-simulated statement.

In [ ]:
!cat README.md

# Home Energy Planner - Neural Network + CSP + Search

Plans a household's next 24 hours of electricity use: a **neural network** forecasts the hourly load, a **CSP** (Backtracking, Forward Checking, AC-3 - all hand-written) schedules flexible appliances, and **Uniform Cost Search / A\*** (hand-written, verified against dynamic programming) plans a home battery. A feedback loop (max. 5 iterations) forbids clashing appliance pairs when the battery plan fails.

> **Real vs simulated.** Consumption data are real (UCI Individual Household Electric Power Consumption, CC BY 4.0). Prices, appliance profiles, battery specification and power limits are **simulated parameters** (`planner/config.py`, `planner/csp.py`). All costs and savings are **simulated**, not measured.

## Project structure
```
planner/config.py        simulated parameters (SimConfig), seeding
planner/data.py          download, hourly aggregation, gap filling, features, chronological split, data dictionary
planner/nn.py      

### 8.2 Presentation and demo run sheet
Minute-by-minute plan for the 10-minute talk and the 5-minute live demo, with likely questions.

In [ ]:
%cd /content/drive/MyDrive/Home_Energy_Planner

!cat docs/PRESENTATION_AND_DEMO.md

/content
# Presentation (10 min) and live demo (5 min) - run sheet

All numbers you quote must come from **your own** `results/*.csv`, `results/nn_vs_baselines.csv` and the demo output. Say "simulated" whenever you mention a cost.

## Before the session (once, ~5 min)
```bash
python make_data.py && python train_nn.py && python run_experiments.py && python run_demo.py --no-show     # warms caches, creates plots
```
Open `outputs/` and `results/` in a file browser as a backup in case the live run fails.

## 10-minute talk (7 slides - the deck in the project folder follows this order)
| min | Slide | What to say | What to show |
|---|---|---|---|
| 0:00-1:30 | Problem and architecture | The planning problem, three techniques, how they cooperate, the feedback loop | architecture diagram |
| 1:30-3:00 | Data: real vs simulated | UCI data (real), prices/limits/appliances/battery (simulated), missing-value strategy, chronological split, no leakage | `data/processed/summary.json` numbers |
| 3

## 9. Project Output Overview

### 9.1 Deliverable folders
Contents of `docs/`, `report/`, `results/` and `outputs/`.

In [ ]:
import os

for folder in ["docs", "report", "results", "outputs"]:
    print(f"\n--- {folder}/ ---")
    if os.path.exists(folder):
        for f in sorted(os.listdir(folder)):
            print(f)
    else:
        print("Folder not found")


--- docs/ ---
PRESENTATION_AND_DEMO.md

--- report/ ---
Home_Energy_Planner_Report.docx

--- results/ ---
exp1_runs.csv
exp1_summary.csv
exp1_ucs_vs_astar.png
exp2_csp_comparison.png
exp2_runs.csv
exp2_summary.csv
exp3_nn_vs_baselines.png
exp3_runs.csv
exp3_summary.csv
nn_forecast_vs_actual.png
nn_training_loss.png
nn_vs_baselines.csv

--- outputs/ ---
demo_csp_pruning.png
demo_nn_on_off.png
demo_plan.png


### 9.2 Result and plot files
Locates all generated result, summary, forecast and plot files in the project.

In [ ]:
%cd /content/drive/MyDrive/Home_Energy_Planner

import os

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    for file in files:
        if any(x in file.lower() for x in [
            "result", "summary", "metric", "plot",
            "forecast", "experiment", "output"
        ]):
            print(os.path.join(root, file))

/content
./run_experiments.py
./planner/plotting.py
./tests/test_forecast.py
./data/processed/summary.json
./models/forecaster.pt
./results/nn_forecast_vs_actual.png
./results/exp1_summary.csv
./results/exp2_summary.csv
./results/exp3_summary.csv
